In [1]:
import pandas as pd 
import numpy as np
import pydantic as py

In [2]:
customer={
    "name":"Sanskruti",
    "age":22,
    "email":"sanskadam7400@gmail.com"
}

In [3]:
from pydantic import BaseModel

class Student(BaseModel):
    name:str
    age:int
    marks:float


s1=Student(
    name="Sanskruti",
    age=22,
    marks=85.5
)

print(s1)

name='Sanskruti' age=22 marks=85.5


In [4]:
from pydantic import ValidationError 
try:
    s2=Student(
    name="Varad",
    age=43,
    marks="hello"
)
except ValidationError  as e:
        print(e)

1 validation error for Student
marks
  Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value='hello', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/float_parsing


In [5]:
class Employee(BaseModel):
    name:str
    age:int
    Salary:float
    dept:str="Not Assigned" #if we dont pass dept yet it will not through an error

e1=Employee(name="V",age=23,Salary=30000)

print(e1)

name='V' age=23 Salary=30000.0 dept='Not Assigned'


In [6]:
class Employee(BaseModel):
    name: str
    age: int


In [7]:
#if you want an employee's age to be between 18 and 60?

#We can use Field().

from pydantic import Field
class emp(BaseModel):
    name:str=Field(max_length=5,min_length=2)
    age:int=Field(ge=18,le=60)
    salary:float


In [8]:
try:
    emp1=emp(name="rr",age=60,salary=2000)
except ValidationError as e:
    print(e)

In [9]:
class employee(BaseModel):
    name:str | None=None
    age:int

e=employee(name="s",age=24)

print(e)

name='s' age=24


In [10]:
class Product(BaseModel):
    name:str
    price:float
    desc:str | None = None
    discount:float=0

In [18]:
# A field validator is a decorator in pydantic that allows us to define 
# custom validation

# The strip() method removes whitespace from both ends of a string. 
# If the string contains only spaces, it becomes empty, and
# our validator raises a ValueError.

from pydantic import field_validator
class Employee(BaseModel):
    name : str

    @field_validator("name")
    @classmethod
    def validate_name(cls,value):
        if not value.strip():
            raise ValueError("Name cannot be empty")
        return value

In [19]:
class Student(BaseModel):
    name:str

    @field_validator("name")
    @classmethod
    def check_name(cls,value):
        if not value.strip():
            raise ValueError("Name cannot be empty")
        return value

s1=Student(name="Sanskruti")
print(s1)

s2=Student(name=" ")
print(s2)

name='Sanskruti'


ValidationError: 1 validation error for Student
name
  Value error, Name cannot be empty [type=value_error, input_value=' ', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

In [27]:
class Employee(BaseModel):
    name:str
    age:int
    salary:float

    @field_validator("name")
    @classmethod
    def check_name(cls,value):
        if not value.strip():
            raise ValueError("Name cannot be Empty !")
        return value

    @field_validator("age")
    @classmethod
    def check_age(cls,value):
        if not value<18:
            raise ValueError("Age must be less than 18 !")
        return value

    @field_validator("salary")
    @classmethod
    def check_sal(cls,value):
        if not value>=0:
            raise ValueError("Salary must be positive !")
        return value

e1=Employee(name="h ",age=2,salary=10384)
print(e1)

name='h ' age=2 salary=10384.0


In [29]:
# Pydantic supports different validator modes. 
# Two important ones are before and after.
# Use before when you need to clean or transform raw input.
# Use after when you want to check a value after type validation.

@field_validator("age", mode="before")
@classmethod
def clean_age(cls, value):
    if isinstance(value, str):
        value = value.strip()
    return value

@field_validator("age", mode="after")
@classmethod
def check_age(cls, value):
    if value < 18:
        raise ValueError("Age must be 18+")
    return value

In [30]:
from pydantic import BaseModel, field_validator

class Product(BaseModel):
    name: str
    price: float

    @field_validator("name", mode="before")
    @classmethod
    def clean_name(cls, value):
        if isinstance(value, str):
            return value.strip()
        return value

    @field_validator("price", mode="after")
    @classmethod
    def check_price(cls, value):
        if value <= 0:
            raise ValueError("Price must be positive")
        return value

p1 = Product(name=" Laptop ", price="50000")
print(p1)

name='Laptop' price=50000.0


In [37]:
# A field_validator validates a single field,
# while a model_validator validates the entire model.
# Types of Model Validators:
# mode="before" Validates raw input before Pydantic validates individual fields.
# mode="after" Validates the model after all fields have been validated.

from pydantic import model_validator
class BankAccount(BaseModel):
    name:str
    balance:float
    withdrawal:float

    @model_validator(mode='after')
    def check_withdrawal(self):
        if self.withdrawal>self.balance:
            raise ValueError("Insufficient Balancce")
        return self

account=BankAccount(name="Sanskruti",balance=10000,withdrawal=2000)

print(account)

account1=BankAccount(name="Sanskruti",balance=10000,withdrawal=12000)

print(account1)

name='Sanskruti' balance=10000.0 withdrawal=2000.0


ValidationError: 1 validation error for BankAccount
  Value error, Insufficient Balancce [type=value_error, input_value={'name': 'Sanskruti', 'ba...00, 'withdrawal': 12000}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

In [39]:
# A nested model is a Pydantic model used 
# as a field inside another Pydantic model.

class Address(BaseModel):
    city:str
    state:str
    pincode:int

class Student(BaseModel):
    name:str
    age:int
    address:Address

student=Student(name="Sanskruti",
                age=22,
                address={
                    "city":"Ichalkaranji",
                    "state":"Maharashtra",
                    "pincode":416115
                }
               )
print(student)

name='Sanskruti' age=22 address=Address(city='Ichalkaranji', state='Maharashtra', pincode=416115)


In [40]:
print(student.name)

Sanskruti


In [41]:
print(student.address.city)

Ichalkaranji


In [42]:
student = Student(
    name="Sanskruti",
    age=21,
    address={
        "city": "Kolhapur",
        "state": "Maharashtra",
        "pincode": "abc"
    }
)

ValidationError: 1 validation error for Student
address.pincode
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='abc', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/int_parsing

In [48]:
# Serialization means converting a Pydantic model into a format that can be 
# stored or shared, such as a Python dictionary or JSON string.

# The model_dump() method converts a Pydantic model into a Python dictionary.

class Student(BaseModel):
    name:str
    age:int
    department:str

student=Student(
    name="Sanskruti",
    age=22,
    department="CSE"
)

print(student.model_dump())

# Output is in python dictionary

# The model_dump_json() method converts a Pydantic model into a JSON string.
print(student.model_dump_json())

{'name': 'Sanskruti', 'age': 22, 'department': 'CSE'}
{"name":"Sanskruti","age":22,"department":"CSE"}


In [49]:
# Deserialization is the process of converting data, 
# such as a dictionary or JSON string, into a Pydantic model.

# Pydantic provides two useful methods:

# model_validate() — 
# creates a model from a Python dictionary or other supported input.

# model_validate_json() — creates a model from a JSON string.

In [51]:
data = {
    "name": "Rahul",
    "age": 22,
    "department": "IT"
}

student = Student.model_validate(data)

print(student)

json_data = '{"name": "Priya", "age": 20, "department": "CSE"}'

student = Student.model_validate_json(json_data)

print(student)

name='Rahul' age=22 department='IT'
name='Priya' age=20 department='CSE'


In [55]:
#Field() is a function provided by Pydantic that allows you to configure model fields.

# Setting constraints using Field()
from pydantic import BaseModel, Field

class Employee(BaseModel):
    name: str = Field(min_length=3, max_length=30)
    age: int = Field(ge=18, le=60)
    salary: float = Field(gt=0)

employee = Employee(
    name="Sanskruti",
    age=21,
    salary=30000
)

print(employee)

name='Sanskruti' age=21 salary=30000.0


In [56]:
# Setting default values

class Product(BaseModel):
    name: str
    price: float
    discount: float = Field(default=0)

product = Product(name="Laptop", price=50000)

print(product)

name='Laptop' price=50000.0 discount=0


In [60]:
# Adding descriptions to 

class Student(BaseModel):
    name: str = Field(description="Full name of the student")
    age: int = Field(description="Age of the student")
    marks: float = Field(ge=0, le=100, description="Marks out of 100")

student=Student(name="sa",age=22,marks=45)

print(student)


name='sa' age=22 marks=45.0


In [61]:
# Combining multiple constraints

class User(BaseModel):
    username: str = Field(
        min_length=5,
        max_length=15,
        description="Unique username"
    )
    age: int = Field(ge=18, le=60)

In [62]:
# ConfigDict allows you to customize how Pydantic handles model data.
# you can:
    # Reject extra fields that aren't defined in the model.
    # Allow field names to be populated using aliases.
    # Make a model immutable.
    # Control how strings are processed.

In [66]:
from pydantic import ConfigDict

class Student(BaseModel):
    model_config=ConfigDict(extra="forbid")

    name:str
    age:int

student = Student(name="Sanskruti", age=21)
print(student)

# extra="forbid" means Pydantic will reject any additional fields that aren't defined in the model.

name='Sanskruti' age=21


In [67]:
student = Student(
    name="Sanskruti",
    age=21,
    city="Kolhapur"
)

ValidationError: 1 validation error for Student
city
  Extra inputs are not permitted [type=extra_forbidden, input_value='Kolhapur', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/extra_forbidden

In [68]:
# extra="ignore"   Ignore extra fields. This is the default.
# extra="forbid"   Raise a validation error for extra fields.
# extra="allow"    Accept and store extra fields.

In [69]:
# You can make a model immutable using frozen=True.
class Student(BaseModel):
    model_confi=ConfigDict(frozen=True)

    name:str
    age:int

student=Student(name="san",age=12)

student.age=22

PydanticUserError: A non-annotated attribute was detected: `model_confi = {'frozen': True}`. All model fields require a type annotation; if `model_confi` is not meant to be a field, you may be able to resolve this error by annotating it as a `ClassVar` or updating `model_config['ignored_types']`.

For further information visit https://errors.pydantic.dev/2.12/u/model-field-missing-annotation

In [70]:
# Using str_strip_whitespace=True:
# This option automatically removes whitespace from the beginning and end of string fields.

In [72]:
class Student(BaseModel):
    model_config = ConfigDict(str_strip_whitespace=True)

    name: str

student = Student(name="  Sanskruti  ")

print(student.name)

Sanskruti


In [73]:
# You can use multiple configuration options in one ConfigDict.
class Student(BaseModel):
    model_config = ConfigDict(
        extra="forbid",
        frozen=True,
        str_strip_whitespace=True
    )

    name: str
    age: int